In [1]:

import os
import warnings
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import log_loss, make_scorer

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[warning] xgboost not installed — skipping XGBoost.")

warnings.filterwarnings("ignore")  # RandomizedSearchCV throws harmless
                                    # convergence warnings on tiny folds

# ----------------------------------------------------------------------
# CONFIG
# ----------------------------------------------------------------------
TRAIN_PATH = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/train_split.csv"
TEST_PATH  = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/Train and test/test_split.csv"
OUTPUT_DIR = "/Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs"

TARGET = "pres.abs"
SPECIES_COL = "Species"
NUMERIC_FEATURES = ["easting", "northing", "disturb", "rainann",
                     "soildepth", "soilfert", "tempann", "topo"]
CLIP = 1e-6

N_ITER_SINGLE = 15   # search budget per species (small data -> smaller budget)
N_ITER_MULTI = 30    # search budget for the pooled model (more data -> bigger budget)
MIN_POS_TO_TUNE = 6  # below this many positives, skip CV search and use defaults


def clipped_log_loss_scorer():
    def _score(y_true, y_pred_proba):
        p = np.clip(y_pred_proba, CLIP, 1 - CLIP)
        return -log_loss(y_true, p, labels=[0, 1])  # negative: higher=better for sklearn
    return make_scorer(_score, needs_proba=True, greater_is_better=True)


NEG_LOG_LOSS = clipped_log_loss_scorer()


# ----------------------------------------------------------------------
# SEARCH SPACES
# ----------------------------------------------------------------------
def get_search_space(model_name):
    if model_name == "LogisticRegression":
        return LogisticRegression(max_iter=3000, class_weight="balanced"), {
            "clf__C": np.logspace(-3, 2, 20),
            "clf__penalty": ["l2"],
        }
    if model_name == "RandomForest":
        return RandomForestClassifier(class_weight="balanced", random_state=42, n_jobs=-1), {
            "clf__n_estimators": [200, 400, 600, 800],
            "clf__max_depth": [None, 4, 6, 8, 12],
            "clf__min_samples_leaf": [1, 2, 3, 5, 10],
            "clf__max_features": ["sqrt", "log2", None],
        }
    if model_name == "MLP":
        return MLPClassifier(max_iter=2000, random_state=42), {
            "clf__hidden_layer_sizes": [(16,), (32,), (32, 16), (64, 32)],
            "clf__alpha": np.logspace(-5, -1, 10),
            "clf__learning_rate_init": [1e-4, 5e-4, 1e-3, 5e-3],
        }
    if model_name == "XGBoost" and HAS_XGB:
        return XGBClassifier(eval_metric="logloss", random_state=42), {
            "clf__n_estimators": [100, 200, 300, 500],
            "clf__max_depth": [2, 3, 4, 6],
            "clf__learning_rate": [0.01, 0.03, 0.05, 0.1],
            "clf__subsample": [0.6, 0.8, 1.0],
            "clf__colsample_bytree": [0.6, 0.8, 1.0],
        }
    raise ValueError(model_name)


def model_names():
    names = ["LogisticRegression", "RandomForest", "MLP"]
    if HAS_XGB:
        names.append("XGBoost")
    return names


def default_model(model_name):
    """Fallback when there isn't enough data to tune (e.g. very rare species)."""
    if model_name == "LogisticRegression":
        return LogisticRegression(max_iter=3000, class_weight="balanced")
    if model_name == "RandomForest":
        return RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                       class_weight="balanced", random_state=42, n_jobs=-1)
    if model_name == "MLP":
        return MLPClassifier(hidden_layer_sizes=(16,), alpha=1e-3, max_iter=1500, random_state=42)
    if model_name == "XGBoost":
        return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.05,
                              eval_metric="logloss", random_state=42)
    raise ValueError(model_name)


def clipped_log_loss(y_true, y_pred):
    y_pred = np.clip(y_pred, CLIP, 1 - CLIP)
    return log_loss(y_true, y_pred, labels=[0, 1])


def tune_pipeline(base_estimator, param_dist, X, y, n_iter, cv_splits):
    pipe = Pipeline([("scale", StandardScaler(with_mean=True)), ("clf", base_estimator)])
    search = RandomizedSearchCV(
        pipe, param_distributions=param_dist, n_iter=n_iter,
        scoring=NEG_LOG_LOSS, cv=cv_splits, random_state=42, n_jobs=-1, error_score=np.nan
    )
    search.fit(X, y)
    return search.best_estimator_, search.best_params_


# ----------------------------------------------------------------------
# (A) SINGLE-SPECIES, tuned per species
# ----------------------------------------------------------------------
def tuned_single_species(train, test, model_name):
    rows = []
    preds_all = np.full(len(test), np.nan)
    base_estimator, param_dist = get_search_space(model_name)

    for species, train_sp in train.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        test_sp = test.loc[mask]
        if test_sp.empty:
            continue
        X_train, y_train = train_sp[NUMERIC_FEATURES], train_sp[TARGET].astype(int)
        X_test = test_sp[NUMERIC_FEATURES]
        y_test = test_sp[TARGET].astype(int)
        n_pos = y_train.sum()

        if n_pos < MIN_POS_TO_TUNE:
            # not enough positives to cross-validate reliably — use defaults
            model = default_model(model_name)
            pipe = Pipeline([("scale", StandardScaler()), ("clf", model)])
            pipe.fit(X_train, y_train)
            best_params = "defaults (too few positives to tune)"
        else:
            n_splits = min(5, n_pos)  # can't have more folds than positives
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
            pipe, best_params = tune_pipeline(
                base_estimator, param_dist, X_train, y_train, N_ITER_SINGLE, cv
            )

        p = pipe.predict_proba(X_test)[:, 1]
        preds_all[mask] = p
        rows.append({
            "species": species, "n_train": len(train_sp), "n_pos_train": int(n_pos),
            "log_loss_single_tuned": clipped_log_loss(y_test, p),
            "best_params_single": str(best_params),
        })

    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    overall = clipped_log_loss(test[TARGET].astype(int), preds_all)
    return df, overall


# ----------------------------------------------------------------------
# (B) MULTI-SPECIES, tuned once on pooled data
# ----------------------------------------------------------------------
def tuned_multi_species(train, test, model_name):
    base_estimator, param_dist = get_search_space(model_name)

    X_train_raw = train[NUMERIC_FEATURES + [SPECIES_COL]]
    y_train = train[TARGET].astype(int)
    X_test_raw = test[NUMERIC_FEATURES + [SPECIES_COL]]
    y_test = test[TARGET].astype(int)

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), [SPECIES_COL]),
    ])
    pipe_template = Pipeline([("prep", preprocessor), ("clf", base_estimator)])
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    search = RandomizedSearchCV(
        pipe_template, param_distributions=param_dist, n_iter=N_ITER_MULTI,
        scoring=NEG_LOG_LOSS, cv=cv, random_state=42, n_jobs=-1, error_score=np.nan
    )
    search.fit(X_train_raw, y_train)
    best_pipe = search.best_estimator_
    p = best_pipe.predict_proba(X_test_raw)[:, 1]
    overall = clipped_log_loss(y_test, p)

    rows = []
    for species, test_sp in test.groupby(SPECIES_COL):
        mask = (test[SPECIES_COL] == species).values
        y_sp = test_sp[TARGET].astype(int)
        p_sp = p[mask]
        rows.append({"species": species, "log_loss_multi_tuned": clipped_log_loss(y_sp, p_sp)})

    df = pd.DataFrame(rows).sort_values("species").reset_index(drop=True)
    return df, overall, search.best_params_


# ----------------------------------------------------------------------
# MAIN
# ----------------------------------------------------------------------
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    train = pd.read_csv(TRAIN_PATH)
    test = pd.read_csv(TEST_PATH)

    print(f"Train: {train.shape}, Test: {test.shape}\n")

    summary_rows = []
    for model_name in model_names():
        print(f"=== Tuning {model_name} ===")
        print("  tuning per-species (single-species approach) ...")
        single_df, single_ll = tuned_single_species(train, test, model_name)

        print("  tuning pooled model (multi-species approach) ...")
        multi_df, multi_ll, multi_best_params = tuned_multi_species(train, test, model_name)

        merged = single_df.merge(multi_df, on="species")
        csv_path = os.path.join(OUTPUT_DIR, f"{model_name}_tuned_per_species.csv")
        merged.to_csv(csv_path, index=False)
        print(f"  [saved] {csv_path}")

        print(f"  Overall (tuned) log-loss  |  single: {single_ll:.4f}   multi: {multi_ll:.4f}   "
              f"({'multi wins' if multi_ll < single_ll else 'single wins'})")
        print(f"  Multi-species best params: {multi_best_params}\n")

        summary_rows.append({
            "model": model_name,
            "overall_log_loss_single_tuned": single_ll,
            "overall_log_loss_multi_tuned": multi_ll,
            "multi_wins_after_tuning": multi_ll < single_ll,
        })

    summary_df = pd.DataFrame(summary_rows)
    summary_path = os.path.join(OUTPUT_DIR, "tuned_summary_across_models.csv")
    summary_df.to_csv(summary_path, index=False)
    print(f"[saved] {summary_path}")

    print("\n===== SUMMARY (tuned single vs tuned multi) =====")
    print(summary_df.to_string(index=False))
    print("\nCompare this table to the untuned summary_across_models.csv from the "
          "original script — if a model's winner flips here, the earlier result was "
          "an artifact of shared hyperparameters, not a real single-vs-multi effect.")


if __name__ == "__main__":
    main()

Train: (4102, 14), Test: (1026, 14)

=== Tuning LogisticRegression ===
  tuning per-species (single-species approach) ...
  tuning pooled model (multi-species approach) ...
  [saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/LogisticRegression_tuned_per_species.csv
  Overall (tuned) log-loss  |  single: 0.5883   multi: 0.6315   (single wins)
  Multi-species best params: {'clf__penalty': 'l2', 'clf__C': 0.001}

=== Tuning RandomForest ===
  tuning per-species (single-species approach) ...
  tuning pooled model (multi-species approach) ...
  [saved] /Users/ak/Documents/UNSW Classes/DATA5925/CODE/single vs multi species/outputs/RandomForest_tuned_per_species.csv
  Overall (tuned) log-loss  |  single: 0.1832   multi: 0.2932   (single wins)
  Multi-species best params: {'clf__n_estimators': 800, 'clf__min_samples_leaf': 1, 'clf__max_features': 'log2', 'clf__max_depth': 8}

=== Tuning MLP ===
  tuning per-species (single-species approach) ...
  tuning poo